# Davinci — GPU and Training Preflight

A diagnostics-only notebook to establish accelerator, memory, dependency, filesystem, and model-size feasibility before training. It does **not** launch training or assume GPU availability.

In [ ]:
import os, sys, platform, json, shutil, subprocess, time
from pathlib import Path
import torch
print('Python:',sys.version.split()[0]); print('Platform:',platform.platform()); print('PyTorch:',torch.__version__)
print('CUDA available:',torch.cuda.is_available()); print('CUDA build:',torch.version.cuda)
print('Disk free (GB):',round(shutil.disk_usage('/content' if Path('/content').exists() else '.').free/1e9,2))
try:
 import psutil
 print('RAM total/free (GB):',round(psutil.virtual_memory().total/1e9,2),round(psutil.virtual_memory().available/1e9,2))
except ImportError: print('Install psutil if RAM diagnostics are desired: pip install psutil')


In [ ]:
gpu_report=[]
if torch.cuda.is_available():
 for i in range(torch.cuda.device_count()):
  p=torch.cuda.get_device_properties(i); gpu_report.append({'index':i,'name':p.name,'memory_GB':round(p.total_memory/1e9,2),'compute_capability':f'{p.major}.{p.minor}','bf16_supported':torch.cuda.is_bf16_supported()})
print(json.dumps(gpu_report,indent=2))
if not torch.cuda.is_available(): print('No CUDA accelerator detected. CPU-only diagnostics are informational; GPU training is not feasible in this runtime.')


## CUDA smoke test

Tests a small allocation, forward/backward pass, and peak allocated memory if CUDA is present. It is not a benchmark of the target model.

In [ ]:
if torch.cuda.is_available():
 device='cuda'; torch.cuda.reset_peak_memory_stats(); a=torch.randn((2048,2048),device=device,requires_grad=True); b=a@a; loss=b.float().mean(); loss.backward(); torch.cuda.synchronize()
 print({'smoke_test':'passed','peak_allocated_GB':round(torch.cuda.max_memory_allocated()/1e9,3),'peak_reserved_GB':round(torch.cuda.max_memory_reserved()/1e9,3)})
 del a,b,loss; torch.cuda.empty_cache()
else: print('CUDA smoke test skipped.')


## Approximate memory arithmetic

These calculations are rough weight/optimizer-state estimates only. They exclude activations, allocator overhead, temporary attention buffers, gradients details, and implementation differences. They must not be used as a substitute for a measured pilot.

In [ ]:
PARAMETERS=508_960_000
for dtype,bytes_per_param in [('fp32',4),('bf16/fp16',2)]: print(dtype,'weight GB',round(PARAMETERS*bytes_per_param/1e9,2))
print('Adam-style parameter, gradient, two moments rough GB:',round(PARAMETERS*(2+2+4+4)/1e9,2),'(BF16 weights+grads and FP32 moments; rough only)')


## Runtime record

Save the report alongside pilot checkpoints; avoid publishing sensitive machine or environment details.

In [ ]:
report={'timestamp_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),'python':sys.version.split()[0],'platform':platform.platform(),'pytorch':torch.__version__,'cuda_build':torch.version.cuda,'cuda_available':torch.cuda.is_available(),'gpus':gpu_report}
if torch.cuda.is_available(): report['cuda_smoke']='passed'
print(json.dumps(report,indent=2))
REPORT_PATH=Path('davinci_preflight_report.json'); REPORT_PATH.write_text(json.dumps(report,indent=2),encoding='utf-8'); print('Saved:',REPORT_PATH.resolve())
